In dit notebook laden we de data in, passen we de gekozen baseline-correctie (Hybrid 4S+ASL) en normalisatie (SNV) methode toe. Vervolgens exporteren we de bewerkte data naar een nieuw CSV-bestand. Hier is voor gekozen, omdat het toepassen van de baseline-correctie op alle spectra alleen al ongeveer 90 minuten duurt. Door de bewerkte data op te slaan, hoeven we deze stap niet telkens opnieuw uit te voeren.

In [1]:
import os
import sys
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Add the parent directory (project root) to Python path
project_root = os.path.abspath('..')  # Go up one level from current notebook
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from utils.baseline_correction_functions import BaselineCorrector
from utils.libs_generic_functions import LibsDataLoader, LibsDataPreprocessor
from utils.normalization_functions import Normalizer

In [2]:
dataloader = LibsDataLoader(data_directory='data')
preprocessor = LibsDataPreprocessor()
corrector = BaselineCorrector()

In [3]:
df = dataloader.load_all_measurements()

MEMORY-EFFICIENT LOADING OF ALL INDIVIDUAL MEASUREMENTS
First pass: Counting measurements...
Processing file 1/316: 2024-11-05T11-26-41_nr-001_B1_tread_aided_10Hz_280A.h5
Processing file 11/316: 2024-11-05T11-51-23_nr-011_B4_innerliner_aided_10Hz_280A.h5
Processing file 21/316: 2024-11-05T12-02-34_nr-021_B7_sidewall_aided_10Hz_280A.h5
Processing file 31/316: 2024-11-05T12-13-30_nr-031_B22_tread_aided_10Hz_280A.h5
Processing file 41/316: 2024-11-05T12-25-42_nr-041_B33_innerliner_aided_10Hz_280A.h5
Processing file 51/316: 2024-11-05T12-36-31_nr-051_B28_sidewall_aided_10Hz_280A.h5
Processing file 61/316: 2024-11-05T12-47-29_nr-061_B15_tread_aided_10Hz_280A.h5
  Counted 10,000 measurements...
Processing file 71/316: 2024-11-05T12-58-45_nr-071_B30_innerliner_aided_10Hz_280A.h5
Processing file 81/316: 2024-11-05T13-17-11_nr-081_B35_innerliner_aided_10Hz_280A.h5
Processing file 91/316: 2024-11-05T14-05-16_nr-091_B26_sidewall_aided_10Hz_280A.h5
Processing file 101/316: 2024-11-05T14-16-52_nr-1

In [4]:
df.head()

,tire_number,origin,measurement_id,184.3,184.4,184.5,184.6,184.7,184.8,184.9,...,981.3,981.4,981.5,981.7,981.8,981.9,982.0,982.1,982.2,982.4
0,1,tread,0,178.5,249.0,195.0,206.5,240.0,228.5,165.0,...,533.0,497.0,549.0,480.0,524.0,480.0,512.0,452.0,534.0,458.0
1,1,tread,1,145.0,208.0,104.0,150.0,232.0,185.0,138.0,...,601.0,550.0,606.0,545.0,550.0,524.0,623.0,529.0,606.0,503.0
2,1,tread,2,190.5,282.0,183.0,217.0,224.0,204.0,176.0,...,533.0,472.0,535.0,486.0,520.0,458.0,536.0,423.0,549.0,422.0
3,1,tread,3,128.0,160.0,118.0,137.0,200.0,162.5,145.0,...,584.0,522.0,603.0,536.0,575.0,532.0,651.0,525.0,624.0,527.0
4,1,tread,4,198.5,237.0,196.0,201.0,301.0,260.0,225.0,...,570.0,491.0,529.0,484.0,541.0,479.0,583.0,484.0,556.0,472.0


In [5]:
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
print(f"Non-feature columns: {non_feature_cols}")

# Get numeric columns (wavelength data)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

Non-feature columns: ['tire_number', 'origin', 'measurement_id']


In [6]:
X = df[wavelength_columns].values
y = df['origin'].values

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Dataset shape: {X.shape}")
print(f"Classes: {label_encoder.classes_}")
print(f"Class distribution: {np.bincount(y_encoded)}")

X_corrected = corrector.apply_correction_batch(X)

# 2. Standardize features
normalizer = Normalizer() 
X_scaled = normalizer.apply_standard_normal_variate_on_dataset(X_corrected)

Dataset shape: (51453, 6699)
Classes: ['innerliner' 'sidewall' 'tread']
Class distribution: [17260 16257 17936]
Applying HYBRID baseline correction to 51453 spectra in batches of 100...
  Processing batch 1/515 (spectra 1-100)


  Processing batch 2/515 (spectra 101-200)
  Processing batch 3/515 (spectra 201-300)
  Processing batch 4/515 (spectra 301-400)
  Processing batch 5/515 (spectra 401-500)
  Processing batch 6/515 (spectra 501-600)
  Processing batch 7/515 (spectra 601-700)
  Processing batch 8/515 (spectra 701-800)
  Processing batch 9/515 (spectra 801-900)
  Processing batch 10/515 (spectra 901-1000)
  Processing batch 11/515 (spectra 1001-1100)
  Processing batch 12/515 (spectra 1101-1200)
  Processing batch 13/515 (spectra 1201-1300)
  Processing batch 14/515 (spectra 1301-1400)
  Processing batch 15/515 (spectra 1401-1500)
  Processing batch 16/515 (spectra 1501-1600)
  Processing batch 17/515 (spectra 1601-1700)
  Processing batch 18/515 (spectra 1701-1800)
  Processing batch 19/515 (spectra 1801-1900)
  Processing batch 20/515 (spectra 1901-2000)
  Processing batch 21/515 (spectra 2001-2100)
  Processing batch 22/515 (spectra 2101-2200)
  Processing batch 23/515 (spectra 2201-2300)
  Processing 

81 minuten

In [7]:
#replace the original data with the processed data
df.head()

,tire_number,origin,measurement_id,184.3,184.4,184.5,184.6,184.7,184.8,184.9,...,981.3,981.4,981.5,981.7,981.8,981.9,982.0,982.1,982.2,982.4
0,1,tread,0,178.5,249.0,195.0,206.5,240.0,228.5,165.0,...,533.0,497.0,549.0,480.0,524.0,480.0,512.0,452.0,534.0,458.0
1,1,tread,1,145.0,208.0,104.0,150.0,232.0,185.0,138.0,...,601.0,550.0,606.0,545.0,550.0,524.0,623.0,529.0,606.0,503.0
2,1,tread,2,190.5,282.0,183.0,217.0,224.0,204.0,176.0,...,533.0,472.0,535.0,486.0,520.0,458.0,536.0,423.0,549.0,422.0
3,1,tread,3,128.0,160.0,118.0,137.0,200.0,162.5,145.0,...,584.0,522.0,603.0,536.0,575.0,532.0,651.0,525.0,624.0,527.0
4,1,tread,4,198.5,237.0,196.0,201.0,301.0,260.0,225.0,...,570.0,491.0,529.0,484.0,541.0,479.0,583.0,484.0,556.0,472.0


In [8]:
processed_data_path = 'data/csv/processed_data.csv'
df.to_csv(processed_data_path, index=False)

In [9]:
# Verify by loading the CSV
import pandas as pd
df = pd.read_csv(processed_data_path)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51453 entries, 0 to 51452
Columns: 6702 entries, tire_number to 982.4
dtypes: float64(6699), int64(2), object(1)
memory usage: 2.6+ GB
